# 04 - Fine-tune Cross-Encoder Re-ranker

Train a Vietnamese CSKH re-ranker using CrossEncoder.

### Purpose
Re-ranker sits **after** FAISS retrieval — it takes top-K candidates from embedding
search and re-scores them with a more accurate cross-attention model.

### Advanced techniques:
- **Cross-Encoder** (cross-attention, more accurate than bi-encoder)
- **Hard negative mining** from FAISS retrieval results
- **Balanced positive/negative pairs** for training stability
- **Google Drive save**

### Requirements
- Colab with **T4 GPU** (~1 compute unit)

### Output
- Fine-tuned CrossEncoder model for backend `reranker.py`

In [ ]:
!pip install -q sentence-transformers>=3.0.0 datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random
import torch
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/vani-copilot'
os.makedirs(DRIVE_DIR, exist_ok=True)
random.seed(42)

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Load knowledge base + training data
from pathlib import Path

knowledge_docs = []
for fname in ['policies.txt', 'faq.txt', 'products.txt']:
    for base in [DRIVE_DIR, '.']:
        p = Path(base) / fname
        if p.exists():
            text = p.read_text(encoding='utf-8')
            paragraphs = [para.strip() for para in text.split('\n\n') if para.strip() and len(para.strip()) > 20]
            knowledge_docs.extend(paragraphs)
            print(f'{fname}: {len(paragraphs)} paragraphs')
            break

train_data = []
for base in [DRIVE_DIR, '.']:
    train_path = os.path.join(base, 'train.jsonl')
    if os.path.exists(train_path):
        with open(train_path, 'r', encoding='utf-8') as f:
            train_data = [json.loads(line) for line in f]
        print(f'Train data: {len(train_data)} samples')
        break

print(f'Total knowledge docs: {len(knowledge_docs)}')

In [ ]:
# Step 1: Use fine-tuned embedding (or base) to generate hard negatives
from sentence_transformers import SentenceTransformer

embed_path = os.path.join(DRIVE_DIR, 'embedding-finetuned')
if os.path.exists(embed_path):
    bi_encoder = SentenceTransformer(embed_path)
    print(f'Loaded fine-tuned embedding from Drive')
else:
    bi_encoder = SentenceTransformer('intfloat/multilingual-e5-base')
    print('Using base embedding (fine-tuned not found on Drive)')

In [ ]:
# Step 2: Build corpus from knowledge base + assistant answers
corpus = list(knowledge_docs)

for item in train_data:
    for msg in item['messages']:
        if msg['role'] == 'assistant' and len(msg['content'].strip()) > 20:
            corpus.append(msg['content'].strip())

# Deduplicate
corpus = list(set(corpus))
print(f'Corpus size: {len(corpus)}')

# Encode all corpus
corpus_embeddings = bi_encoder.encode(
    [f'passage: {d}' for d in corpus],
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)
print(f'Corpus encoded: {corpus_embeddings.shape}')

In [ ]:
# Step 3: Generate (query, passage, label) with hard negatives
# For each user question, the correct answer is positive (label=1)
# A top-K retrieved (but wrong) document is the hard negative (label=0)

from sentence_transformers import InputExample

train_samples = []

for item in train_data:
    msgs = item['messages']
    for i in range(len(msgs) - 1):
        if msgs[i]['role'] == 'user' and msgs[i+1]['role'] == 'assistant':
            query = msgs[i]['content'].strip()
            positive = msgs[i+1]['content'].strip()
            if len(query) < 10 or len(positive) < 15:
                continue

            # Positive pair
            train_samples.append(InputExample(texts=[query, positive], label=1.0))

            # Hard negative: retrieve top-5, pick the best non-matching one
            q_emb = bi_encoder.encode(f'query: {query}', normalize_embeddings=True)
            scores = q_emb @ corpus_embeddings.T
            top_indices = np.argsort(scores)[-10:][::-1]

            for idx in top_indices:
                candidate = corpus[idx]
                if candidate != positive and len(candidate) > 15:
                    train_samples.append(InputExample(texts=[query, candidate], label=0.0))
                    break

    # Limit encoding time
    if len(train_samples) > 20000:
        break

random.shuffle(train_samples)

n_pos = sum(1 for s in train_samples if s.label > 0.5)
n_neg = len(train_samples) - n_pos
print(f'Total samples: {len(train_samples)}')
print(f'Positive: {n_pos}, Negative: {n_neg}')
print(f'Ratio: {n_pos / max(n_neg, 1):.2f}')

In [ ]:
# Step 4: Train CrossEncoder
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

# Use a multilingual base model
cross_encoder = CrossEncoder(
    'xlm-roberta-base',
    num_labels=1,
    max_length=256,
)

# Split for eval
eval_size = min(500, len(train_samples) // 10)
eval_samples = train_samples[:eval_size]
fit_samples = train_samples[eval_size:]

train_dataloader = DataLoader(fit_samples, shuffle=True, batch_size=32)

# Build evaluator
eval_sentences1 = [s.texts[0] for s in eval_samples]
eval_sentences2 = [s.texts[1] for s in eval_samples]
eval_labels = [int(s.label) for s in eval_samples]

evaluator = CEBinaryClassificationEvaluator(
    sentence_pairs=list(zip(eval_sentences1, eval_sentences2)),
    labels=eval_labels,
    name='reranker_eval',
)

print(f'Train: {len(fit_samples)} samples')
print(f'Eval: {len(eval_samples)} samples')
print(f'Starting training...')

cross_encoder.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=5,
    warmup_steps=100,
    evaluation_steps=200,
    output_path='./reranker-finetuned',
    show_progress_bar=True,
)

print('Re-ranker training complete!')

In [ ]:
# Step 5: Test re-ranker
test_queries = [
    ('đổi trả hàng bị lỗi', 'Hàng bị lỗi sản xuất được đổi miễn phí trong 7 ngày'),
    ('đổi trả hàng bị lỗi', 'Áo croptop mới về chất liệu cotton 100%'),
    ('ship bao lâu', 'Giao hàng nội thành 1-2 ngày, ngoại thành 3-5 ngày'),
    ('ship bao lâu', 'Mã giảm giá VANI50 giảm 50k cho đơn từ 500k'),
]

for q, d in test_queries:
    score = cross_encoder.predict([(q, d)])[0]
    label = 'RELEVANT' if score > 0.5 else 'NOT relevant'
    print(f'[{score:.3f}] {label} | Q: "{q}" | D: "{d[:60]}..."')

In [ ]:
# Save to Drive
import shutil

save_path = os.path.join(DRIVE_DIR, 'reranker-finetuned')
if os.path.exists('./reranker-finetuned'):
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    shutil.copytree('./reranker-finetuned', save_path)

print(f'Saved to: {save_path}')
print('Done! Download for backend integration.')